# Scientific validation of the deployment-5 rotation figure

These checks preserve the scientific evidence from the completed exploration; intermediate figure drafts and presentation experiments were retired. The final figure remains in `beam_modulation.ipynb`.

Run from within this repository. Inputs are the final notebook's sidecar and filtered deployment-5 data. The first three checks additionally require `hera_filters` installed in the kernel. Figure-dependent checks run the final notebook once and reuse its reduction, regenerating its exports. All generated outputs remain ignored.

Recorded exploration results (not newly measured by this cleanup): transmitter-off prediction error was about 1.6% for the flanking median versus 2.5% for log-power DPSS and 28% for linear-power fitting; static bias correction did not help. The two comb residues trade coupling with azimuth; residue 8 led residue 0 by 7.9 dB at the selected -90-degree azimuth. Upper-band structure justified removing the smoothness rejection of 224.6 MHz. FM emission rotates with the antenna and cannot be separated from the injected tone by continuum subtraction. The ground-control threshold and its blind spots are evaluated below.

The final selection was 42 tones over 56.6–240.2 MHz, with continuum-subtracted depths 8.1–33.1 dB (median 25.4). These numbers are provenance from the exploration commits, not acceptance thresholds.


In [ ]:
from pathlib import Path
import json
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

REPO = Path.cwd().resolve()
while not (REPO / "notebooks/christian").is_dir():
    if REPO == REPO.parent:
        raise FileNotFoundError("Run from within the data-analysis repository")
    REPO = REPO.parent
OUT = REPO / "notebooks/christian/beam_modulation"
__file__ = str(OUT / "validation.py")  # legacy diagnostics resolve outputs beside this notebook
_figure = None

def load_figure():
    """Run the final notebook's code in its own namespace, once per validation session."""
    global _figure
    if _figure is None:
        notebook = json.loads((OUT / "beam_modulation.ipynb").read_text())
        namespace = {"__name__": "__validation__"}
        previous = Path.cwd()
        try:
            os.chdir(OUT)
            for index, cell in enumerate(notebook["cells"]):
                if cell["cell_type"] == "code":
                    exec(compile("".join(cell["source"]), f"final-notebook-cell-{index}", "exec"), namespace)
            _figure = namespace
        finally:
            os.chdir(previous)
            plt.close("all")
    return _figure


## Continuum estimator: transmitter-off truth

Preserved from `explore/45_gap_validation.py`; diagnostic calculations retained.

In [ ]:
"""How accurately can the continuum under a comb channel be predicted?

The comb transmitter was off until ~18:14 UTC on 2026-07-17, so in the earlier
files the comb channels contain continuum only and their true value is known.
That is the exact prediction task the subtraction needs, with the real mask
geometry (3 masked channels, nearest usable channel two away).

Compares a global DPSS model (hera_filters) against the flanking median used by
the current figure.
"""
import sys
from pathlib import Path

import h5py
import numpy as np


from hera_filters import dspec  # noqa: E402

DATA = REPO / "data/deployment5_filtered"
OFF = ["corr_20260717_174008Z.h5", "corr_20260717_174217Z.h5", "corr_20260717_174426Z.h5",
       "corr_20260717_174635Z.h5", "corr_20260717_174844Z.h5"]
COMB_RESIDUE, COMB_SPILL = 8, (7, 8, 9, 15, 0, 1)
FM_BAND, FIT_BAND = (86.0, 110.0), (48.0, 235.0)

blocks = []
for n in OFF:
    with h5py.File(DATA / n, "r") as f:
        d = f["data/4"][:].astype(np.float64)
        blocks.append(d)
d = np.concatenate(blocks)
d[d <= 0] = np.nan
nchan = d.shape[1]
chan = np.arange(nchan)
freq = chan * 0.244140625
print(f"transmitter-off data: {d.shape[0]} integrations x {nchan} channels")

in_band = (freq > FIT_BAND[0]) & (freq < FIT_BAND[1])
is_fm = (freq > FM_BAND[0]) & (freq < FM_BAND[1])
clean = in_band & ~np.isin(chan % 16, COMB_SPILL) & ~is_fm & np.isfinite(d).all(0)
tones = chan[(chan % 16 == COMB_RESIDUE) & in_band & ~is_fm & np.isfinite(d).all(0)]

RADIOM = 1.0 / np.sqrt(0.244140625e6 * 0.537)
xb, db, cleanb = freq[in_band] * 1e6, d[:, in_band], clean[in_band]
tb = np.searchsorted(chan[in_band], tones)

# confirm the comb really is off
snr = np.nanmedian(d[:, tones], 0) / np.array(
    [np.nanmedian(np.nanmedian(d[:, [c + o for o in range(-6, 7)
                                     if 3 <= abs(o) <= 6 and clean[c + o]]], 1)) for c in tones])
print(f"tone/continuum with transmitter off: median {10*np.log10(np.median(snr)):+.3f} dB "
      f"(range {10*np.log10(snr.min()):+.2f} to {10*np.log10(snr.max()):+.2f})\n")

print(" estimator                err_at_tone_chan   vs_radiometer")
for tau_ns in (50.0, 100.0, 200.0):
    basis, _ = dspec.dpss_operator(xb, [0.0], [tau_ns * 1e-9], eigenval_cutoff=[1e-9])
    basis = np.real(basis)
    model = (basis @ (np.linalg.pinv(basis[cleanb]) @ db[:, cleanb].T)).T
    err = np.nanmedian(np.abs(model[:, tb] / db[:, tb] - 1.0))
    print(f" DPSS {tau_ns:5.0f} ns ({basis.shape[1]:3d} modes)  {100*err:10.3f} %   {err/RADIOM:10.2f}x")

fl = np.array([np.nanmedian(d[:, [c + o for o in range(-6, 7)
                                  if 3 <= abs(o) <= 6 and clean[c + o]]], 1) for c in tones]).T
err_fl = np.nanmedian(np.abs(fl / d[:, tones] - 1.0))
print(f" flanking median           {100*err_fl:10.3f} %   {err_fl/RADIOM:10.2f}x")
np.savez_compressed(Path(__file__).parent / "gap_validation.npz",
                    tone_freq=freq[tones], err_flanking=err_fl)


## Continuum estimator: fitting variants

Preserved from `explore/46_dpss_variants.py`; diagnostic calculations retained.

In [ ]:
"""Why the global DPSS fit loses to a 13-channel median, and whether it can be fixed.

Same transmitter-off truth as 45. Varies three things: fitting in linear power vs
log power, fitting across the FM hole vs in sub-bands either side of it, and the
delay half-width (capped by the 256 ns alias of the periodic comb mask).
"""
import sys
from pathlib import Path

import h5py
import numpy as np


from hera_filters import dspec  # noqa: E402

DATA = REPO / "data/deployment5_filtered"
OFF = ["corr_20260717_174008Z.h5", "corr_20260717_174217Z.h5", "corr_20260717_174426Z.h5",
       "corr_20260717_174635Z.h5", "corr_20260717_174844Z.h5"]
COMB_RESIDUE, COMB_SPILL, FM_BAND = 8, (7, 8, 9, 15, 0, 1), (86.0, 110.0)

d = np.concatenate([h5py.File(DATA / n, "r")["data/4"][:].astype(np.float64) for n in OFF])
d[d <= 0] = np.nan
chan = np.arange(d.shape[1])
freq = chan * 0.244140625
RADIOM = 1.0 / np.sqrt(0.244140625e6 * 0.537)
finite = np.isfinite(d).all(0)
is_fm = (freq > FM_BAND[0]) & (freq < FM_BAND[1])


def run(lo, hi, tau_ns, logfit):
    band = (freq > lo) & (freq < hi) & finite
    cl = band & ~np.isin(chan % 16, COMB_SPILL) & ~is_fm
    tn = band & (chan % 16 == COMB_RESIDUE) & ~is_fm
    if tn.sum() < 3:
        return None
    xb = freq[band] * 1e6
    y = d[:, band]
    y = np.log10(y) if logfit else y
    basis, _ = dspec.dpss_operator(xb, [0.0], [tau_ns * 1e-9], eigenval_cutoff=[1e-9])
    basis = np.real(basis)
    m = cl[band]
    model = (basis @ (np.linalg.pinv(basis[m]) @ y[:, m].T)).T
    if logfit:
        model = 10 ** model
    t = tn[band]
    return np.nanmedian(np.abs(model[:, t] / d[:, tn] - 1.0)), basis.shape[1], int(tn.sum())


print(" fit space  band            tau_ns  modes  tones   err_at_tone   vs_radiom")
for logfit in (False, True):
    for label, bands in (("48-235 (over FM)", [(48.0, 235.0)]),
                         ("50-86 + 110-200", [(50.0, 86.0), (110.0, 200.0)])):
        for tau_ns in (100.0, 200.0, 300.0):
            errs, nm, nt = [], [], 0
            for lo, hi in bands:
                r = run(lo, hi, tau_ns, logfit)
                if r:
                    errs.append(r[0] * r[2]); nm.append(r[1]); nt += r[2]
            if not errs:
                continue
            e = sum(errs) / nt
            print(f" {'log10' if logfit else 'linear':<9}  {label:<15} {tau_ns:6.0f}  "
                  f"{'+'.join(str(v) for v in nm):>5}  {nt:5d}  {100*e:10.3f} %  {e/RADIOM:9.2f}x")

fl = np.array([np.nanmedian(d[:, [c + o for o in range(-6, 7) if 3 <= abs(o) <= 6
                                  and finite[c + o] and (c + o) % 16 not in COMB_SPILL]], 1)
               for c in chan[(chan % 16 == COMB_RESIDUE) & (freq > 50) & (freq < 200) & ~is_fm]]).T
tt = chan[(chan % 16 == COMB_RESIDUE) & (freq > 50) & (freq < 200) & ~is_fm]
e = np.nanmedian(np.abs(fl / d[:, tt] - 1.0))
print(f"\n flanking median (current method)                        {100*e:10.3f} %  {e/RADIOM:9.2f}x")


## Continuum estimator: split-half bias correction

Preserved from `explore/47_bias_correction.py`; diagnostic calculations retained.

In [ ]:
"""Both continuum estimators are systematics-limited, not noise-limited.

45/46 measured 1.6% (flanking median) and 2.5% (DPSS in log power) error at the
comb channels, against a 0.28% radiometer noise. If that error is a static
property of the bandpass, the transmitter-off window measures it and it can be
divided out. Split-half within the off window tests whether it is stable enough
to be worth applying.
"""
import sys
from pathlib import Path

import h5py
import numpy as np


from hera_filters import dspec  # noqa: E402

DATA = REPO / "data/deployment5_filtered"
OFF = ["corr_20260717_174008Z.h5", "corr_20260717_174217Z.h5", "corr_20260717_174426Z.h5",
       "corr_20260717_174635Z.h5", "corr_20260717_174844Z.h5"]
COMB_RESIDUE, COMB_SPILL, FM_BAND = 8, (7, 8, 9, 15, 0, 1), (86.0, 110.0)

d = np.concatenate([h5py.File(DATA / n, "r")["data/4"][:].astype(np.float64) for n in OFF])
d[d <= 0] = np.nan
chan = np.arange(d.shape[1])
freq = chan * 0.244140625
RADIOM = 1.0 / np.sqrt(0.244140625e6 * 0.537)
finite = np.isfinite(d).all(0)
is_fm = (freq > FM_BAND[0]) & (freq < FM_BAND[1])
band = (freq > 48.0) & (freq < 235.0) & finite
clean = band & ~np.isin(chan % 16, COMB_SPILL) & ~is_fm
tones = chan[band & (chan % 16 == COMB_RESIDUE) & ~is_fm]
n = d.shape[0] // 2
print(f"{d.shape[0]} off integrations, split {n}/{d.shape[0]-n}, {len(tones)} comb channels\n")


def flanking_est(data):
    return np.array([np.nanmedian(data[:, [c + o for o in range(-6, 7) if 3 <= abs(o) <= 6
                                           and clean[c + o]]], 1) for c in tones]).T


def dpss_est(data, tau_ns=100.0):
    xb = freq[band] * 1e6
    y = np.log10(data[:, band])
    basis, _ = dspec.dpss_operator(xb, [0.0], [tau_ns * 1e-9], eigenval_cutoff=[1e-9])
    basis = np.real(basis)
    m = clean[band]
    model = 10 ** (basis @ (np.linalg.pinv(basis[m]) @ y[:, m].T)).T
    return model[:, np.searchsorted(chan[band], tones)]


truth = d[:, tones]
print(" estimator          raw_err   bias-corrected   vs_radiometer   bias spread")
for name, fn in (("flanking median", flanking_est), ("DPSS log 100 ns", dpss_est)):
    est = fn(d)
    raw = np.nanmedian(np.abs(est / truth - 1.0))
    bias = np.nanmedian(est[:n] / truth[:n], axis=0)          # trained on first half
    corr = np.nanmedian(np.abs(est[n:] / bias / truth[n:] - 1.0))   # tested on second
    print(f" {name:<17} {100*raw:6.3f} %   {100*corr:10.3f} %   {corr/RADIOM:10.2f}x   "
          f"{10*np.log10(bias.min()):+.2f} to {10*np.log10(bias.max()):+.2f} dB")

# is the residual after correction random or still structured?
est = flanking_est(d)
bias = np.nanmedian(est[:n] / truth[:n], axis=0)
r = est[n:] / bias / truth[n:] - 1.0
print(f"\n flanking, corrected: per-integration scatter {100*np.nanstd(r):.3f} %, "
      f"time-median residual per channel {100*np.nanmedian(np.abs(np.nanmedian(r, 0))):.3f} %")
print(f" radiometer noise on a 13-channel median estimate ~ {100*RADIOM*np.sqrt(1+1/13.):.3f} %")


## Polarization coupling across azimuth

Preserved from `explore/54_pol_alignment.py`; diagnostic calculations retained.

In [ ]:
"""Which comb is aligned with the bowtie at the azimuth of the figure?

The transmitter is dual-polarization and its two polarizations sit on alternating
comb channels: residue 8 and residue 0. The suspended bowtie is single
polarization, so as the platform azimuth steps between rotations the coupling to
the two transmitter polarizations should trade off. If instead one comb is
uniformly weaker at every azimuth, the difference is transmit power, not
alignment.
"""
import os, runpy
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


g = load_figure()
d4, chan, freq, el, az = g["d4"], g["chan"], g["freq"], g["el"], g["az"]
to_db, is_fm, finite = g["to_db"], g["is_fm"], g["finite"]
rotations, rot_az, ROT = g["rotations"], g["rot_az"], g["ROT"]
COMB_SPILL = g["COMB_SPILL"]


def flank(c):
    return [c + o for o in range(-6, 7) if 3 <= abs(o) <= 6 and 0 <= c + o < len(chan)
            and finite[c + o] and (c + o) % 16 not in COMB_SPILL and not is_fm[c + o]]


sets = {}
for res in (8, 0):
    sets[res] = [c for c in chan[(chan % 16 == res) & (freq > 56) & (freq < 221)]
                 if finite[c] and not is_fm[c] and flank(c)]
    print(f"residue {res}: {len(sets[res])} candidate tones")

rows = []
for r, (a, b) in enumerate(rotations):
    pk = np.zeros(len(el), bool)
    pk[a:b] = True
    pk &= np.abs(el) < 30.0
    if pk.sum() < 5:
        continue
    vals = {}
    for res, cs in sets.items():
        vals[res] = np.median([to_db(np.nanmedian(d4[pk][:, c]))
                               - to_db(np.nanmedian(d4[pk][:, flank(c)])) for c in cs])
    rows.append((r, rot_az[r], vals[8], vals[0]))

rows = np.array(rows)
good = rows[rows[:, 0] <= 29]          # well-behaved first half of the raster
fig, ax = plt.subplots(figsize=(6.4, 3.4))
ax.plot(good[:, 1], good[:, 2], "o-", ms=3, lw=1, label="residue 8")
ax.plot(good[:, 1], good[:, 3], "s-", ms=3, lw=1, label="residue 0")
ax.axvline(-90, color="0.5", ls="--", lw=1, label="azimuth of the figure")
ax.set_xlabel("Platform azimuth [deg]")
ax.set_ylabel("Median tone / continuum at peak [dB]")
ax.set_title("Coupling of the two transmitter polarizations vs azimuth", fontsize=10)
ax.legend(fontsize=8); ax.grid(alpha=0.3)
fig.tight_layout()
out = Path(__file__).parent / "54_pol_alignment.png"
fig.savefig(out, dpi=200, bbox_inches="tight")

i = int(np.argmin(np.abs(good[:, 1] - (-90.0))))
print(f"\nat the figure's azimuth ({good[i,1]:+.0f} deg): "
      f"residue 8 = {good[i,2]:.1f} dB, residue 0 = {good[i,3]:.1f} dB "
      f"-> residue {'8' if good[i,2] > good[i,3] else '0'} is stronger by "
      f"{abs(good[i,2]-good[i,3]):.1f} dB")
print(f"across azimuth: residue 8 spans {good[:,2].min():.1f}-{good[:,2].max():.1f} dB, "
      f"residue 0 spans {good[:,3].min():.1f}-{good[:,3].max():.1f} dB")
print(f"difference (8 - 0) spans {np.min(good[:,2]-good[:,3]):+.1f} to "
      f"{np.max(good[:,2]-good[:,3]):+.1f} dB, "
      f"correlation with azimuth {np.corrcoef(good[:,1], good[:,2]-good[:,3])[0,1]:+.2f}")
print(f"wrote {out}")


## Tone selection: rejected channels

Preserved from `explore/55_rejected_curves.py`; diagnostic calculations retained.

In [ ]:
"""Every comb tone that the figure throws away, and why."""
import os, runpy
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


g = load_figure()
d4, chan, freq, centres = g["d4"], g["chan"], g["freq"], g["centres"]
binned, flanking, excess_profile = g["binned"], g["flanking"], g["excess_profile"]
to_db, is_fm, finite = g["to_db"], g["is_fm"], g["finite"]
A, B, ZERO, NSIG = g["A"], g["B"], g["ZERO"], g["NSIG"]
MAX_ROUGHNESS, COMB_RESIDUE = g["MAX_ROUGHNESS"], g["COMB_RESIDUE"]
kept = set(g["tones"])

groups = {"no data": [], "flanking channels RFI-rough": [],
          "tone not significant": [], "FM band, no clean flanks": []}
core = np.abs(centres) < 60
for c in chan[(chan % 16 == COMB_RESIDUE) & (freq > 50) & (freq < 250)]:
    if c in kept:
        continue
    if is_fm[c]:
        # inside the FM band every neighbour is excluded too, so there is no
        # continuum reference and no excess can be formed -- nothing to draw
        groups["FM band, no clean flanks"].append((c, None))
        continue
    if not finite[c] or not flanking(c):
        groups["no data / no clean flanks"].append((c, None)); continue
    prof, ok = excess_profile(c)
    nb = flanking(c)
    rough = np.nanmedian(np.abs(np.diff(binned(to_db(np.nanmedian(d4[:, nb], 1)), A, B), 2)))
    if rough >= MAX_ROUGHNESS:
        groups["flanking channels RFI-rough"].append((c, prof)); continue
    groups["tone not significant"].append((c, prof))

plot = {k: v for k, v in groups.items() if v}
fig, axes = plt.subplots(1, len(plot), figsize=(3.6 * len(plot), 3.3), sharey=True)
axes = np.atleast_1d(axes)
for ax, (name, items) in zip(axes, plot.items()):
    drawn = 0
    for c, prof in items:
        if prof is None or not np.any(np.isfinite(prof)):
            continue
        ax.plot(centres, prof, lw=0.9, label=f"{freq[c]:.0f}")
        drawn += 1
    if drawn == 0:
        ax.text(0.5, 0.5, "no continuum reference,\nnothing to plot:\n"
                + ", ".join(f"{freq[c]:.0f}" for c, _ in items) + " MHz",
                ha="center", va="center", transform=ax.transAxes, fontsize=8, color="0.35")
    ax.axhline(0.0, color="0.75", lw=0.5, ls=":")
    ax.set_xlim(-180, 180); ax.set_ylim(-35, 8)
    ax.set_xticks([-180, -90, 0, 90, 180]); ax.grid(alpha=0.25, lw=0.4)
    ax.set_title(f"{name}\n({len(items)} tones)", fontsize=9)
    ax.set_xlabel("Platform rotation angle [deg]", fontsize=8)
    ax.tick_params(labelsize=7)
    if drawn:
        ax.legend(fontsize=6, ncol=2, title="MHz", title_fontsize=6, loc="lower center")
axes[0].set_ylabel("Injected tone power rel. $0^\\circ$ [dB]", fontsize=8)
nodata = sum(1 for v in groups.values() for _, p in v
             if p is None or not np.any(np.isfinite(p)))
fig.suptitle(f"{len(kept)} of {len(kept) + sum(len(v) for v in groups.values())} in-band comb "
             f"channels kept; {nodata} rejected channels have no continuum reference and so no curve", fontsize=9, y=1.04)
fig.tight_layout()
out = Path(__file__).parent / "55_rejected_curves.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
print(f"kept {len(kept)} tones; rejected:")
for k, v in groups.items():
    fr = ", ".join(f"{freq[c]:.0f}" for c, _ in v)
    print(f"  {k:<28} {len(v):2d}   {fr}")
print(f"wrote {out}")


## Upper-band response structure

Preserved from `explore/57_top_band.py`; diagnostic calculations retained.

In [ ]:
"""Does the secondary dip at the top of the band evolve smoothly with frequency?

A smoothness cut on each tone's own profile was briefly used to reject 224.6 MHz,
on the assumption that its split null was a channel glitch. Plotting the whole
top of the band together shows the same secondary dip near -45 deg in every tone
from 201 to 240 MHz, deepening progressively with frequency, so it is structure
in the response and the cut was discarding it. The cut was removed.
"""
import os
import runpy
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402


g = load_figure()
chan, freq, centres = g["chan"], g["freq"], g["centres"]
excess_profile, finite, is_fm = g["excess_profile"], g["finite"], g["is_fm"]
kept = set(g["tones"])

top = [c for c in chan[(chan % 16 == 8) & (freq > 200) & (freq < 245)]
       if finite[c] and not is_fm[c]]
fig, ax = plt.subplots(figsize=(7, 3.6))
for c in top:
    prof, _ = excess_profile(c)
    ax.plot(centres, prof, "-" if c in kept else "--", lw=1.1,
            label=f"{freq[c]:.0f} MHz{'' if c in kept else '  (rejected)'}")
ax.set_xlim(-180, 180)
ax.set_ylim(-30, 5)
ax.grid(alpha=0.3)
ax.set_xticks([-180, -90, 0, 90, 180])
ax.set_xlabel("Platform rotation angle [deg]")
ax.set_ylabel("rel. $0^\\circ$ [dB]")
ax.set_title("Top of the band: does the split null evolve smoothly?", fontsize=10)
ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
out = Path(__file__).parent / "57_top_band.png"
fig.savefig(out, dpi=200, bbox_inches="tight")

core = np.abs(centres) < 60
for c in top:
    prof, _ = excess_profile(c)
    print(f"  {freq[c]:6.1f} MHz  {'kept' if c in kept else 'rejected':>8}  "
          f"max kink {np.nanmax(np.abs(np.diff(prof[core], 2))):.1f} dB")
print(f"wrote {out}")


## FM-band contamination

Preserved from `explore/60_fm_channels.py`; diagnostic calculations retained.

In [ ]:
"""Can the FM-band comb channels be recovered, or is the tone simply buried?

Two questions. (i) Is the injected tone detectable at all in the FM band? The
transmitter was off before ~18:14 UTC, so comparing the comb channel against its
local neighbours with the transmitter off and again during the raster separates
"tone present but continuum hard to estimate" from "tone buried under FM".
(ii) What do the raw rotation profiles look like there?
"""
import os, runpy
from pathlib import Path

import h5py
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


g = load_figure()
d4, chan, freq, centres = g["d4"], g["chan"], g["freq"], g["centres"]
binned, to_db, is_fm, A, B, ZERO = g["binned"], g["to_db"], g["is_fm"], g["A"], g["B"], g["ZERO"]
el, COMB_SPILL = g["el"], g["COMB_SPILL"]

DATA = REPO / "data/deployment5_filtered"
OFF = ["corr_20260717_174008Z.h5", "corr_20260717_174217Z.h5", "corr_20260717_174426Z.h5",
       "corr_20260717_174635Z.h5", "corr_20260717_174844Z.h5"]
off = np.concatenate([h5py.File(DATA / n, "r")["data/4"][:].astype(np.float64) for n in OFF])
off[off <= 0] = np.nan

fm_tones = chan[(chan % 16 == 8) & is_fm]
ref_tones = [c for c in g["tones"] if 110 < freq[c] < 145][:4]

def local(c, arr, allow_fm):
    nb = [c + o for o in range(-6, 7) if 3 <= abs(o) <= 6 and 0 <= c + o < len(chan)
          and (c + o) % 16 not in COMB_SPILL and (allow_fm or not is_fm[c + o])]
    return np.nanmedian(np.nanmedian(arr[:, nb], axis=1))

pk = np.zeros(len(el), bool); pk[A:B] = True; pk &= np.abs(el) < 30
print("  freq      tone/local, OFF     tone/local, ON      rise when transmitter on")
for c in list(fm_tones) + ref_tones:
    a = to_db(np.nanmedian(off[:, c])) - to_db(local(c, off, True))
    b = to_db(np.nanmedian(d4[pk][:, c])) - to_db(local(c, d4[pk], True))
    tag = "  FM" if is_fm[c] else "  ref"
    print(f" {freq[c]:6.1f}  {a:14.2f} dB {b:16.2f} dB {b-a:19.2f} dB{tag}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.0, 3.4), sharey=True)
for c in fm_tones:
    a1.plot(centres, binned(to_db(d4[:, c]), A, B) - np.nanmedian(binned(to_db(d4[:, c]), A, B)[ZERO-1:ZERO+2]),
            lw=1.0, label=f"{freq[c]:.0f}")
a1.set_title("FM-band comb channels, raw", fontsize=9); a1.legend(fontsize=6, ncol=2)
for c in ref_tones:
    a2.plot(centres, binned(to_db(d4[:, c]), A, B) - np.nanmedian(binned(to_db(d4[:, c]), A, B)[ZERO-1:ZERO+2]),
            lw=1.0, label=f"{freq[c]:.0f}")
a2.set_title("kept tones just above the FM band, raw", fontsize=9); a2.legend(fontsize=6)
for ax in (a1, a2):
    ax.set_xlim(-180, 180); ax.set_xticks([-180, -90, 0, 90, 180]); ax.grid(alpha=0.3)
    ax.set_xlabel("Platform rotation angle [deg]", fontsize=8); ax.tick_params(labelsize=7)
a1.set_ylabel("Raw power rel. $0^\\circ$ [dB]", fontsize=8)
fig.tight_layout()
out = Path(__file__).parent / "60_fm_channels.png"
fig.savefig(out, dpi=200, bbox_inches="tight")
print(f"\nwrote {out}")


## Ground-control threshold and its limits

Preserved from `explore/61_ground_copy_cut.py`; diagnostic calculations retained.

In [ ]:
"""Which tones' continuum estimates are safe to plot as sky modulation.

The rotation figure gains a second panel showing the continuum that is subtracted
from each tone -- the sky and receiver power, carried by the channels with no
injected signal in them. Plotted raw it is wrong: a dozen channels between 110
and 240 MHz swing 1-3 dB through the turn, far more than the sky does, because
they carry a terrestrial transmitter rather than sky.

The control is the stationary ground copy. It does not rotate, so anything it
also saw change over the same integrations changed in time rather than with
antenna orientation. A tone's continuum is plotted only where the ground copy is
steady on the same flanking channels.

This script fixes the threshold, shows that the quoted median does not depend on
it, and records the one thing the test cannot do.

Reduction is identical to build_nb.py; only the continuum panel is new.
"""
from pathlib import Path

import h5py
import matplotlib
import numpy as np

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

REPO = REPO
SIDECAR = REPO / "notebooks/christian/deployment5/motor_scan_20260717_key4.h5"
KEY0 = REPO / "notebooks/christian/beam_modulation/key0_raster.npz"
OUT = Path(__file__).resolve().parent

COMB_RESIDUE = 8
COMB_SPILL = (7, 8, 9, 15, 0, 1)
FM_BAND = (86.0, 110.0)
BIN_DEG = 5.0
MAX_ROUGHNESS = 0.25
NSIG = 3.0

with h5py.File(SIDECAR, "r") as f:
    d4 = f["data4"][:].astype(np.float64)
    t = f["times"][:]
    deg = 1.0 / f.attrs["counts_per_deg"]
    az = f["az_counts"][:] * deg
    el = f["el_counts"][:] * deg
    freq = f["freqs"][:]

d4[d4 <= 0] = np.nan
nchan = d4.shape[1]
chan = np.arange(nchan)
is_fm = (freq > FM_BAND[0]) & (freq < FM_BAND[1])

d0 = load_figure()["d0"].copy()
d0[d0 <= 0] = np.nan

turns = np.where(np.diff(np.sign(np.diff(el))) != 0)[0] + 1
rotations = [(a, b) for a, b in zip(np.r_[0, turns], np.r_[turns, len(el)]) if b - a > 100]
rot_az = np.array([np.nanmedian(az[a:b]) for a, b in rotations])
ROT = int(np.argmin(np.abs(rot_az - (-90.0))))
A, B = rotations[ROT]
finite = np.isfinite(d4[A:B]).mean(0) > 0.9

edges = np.arange(-180.0, 180.0 + BIN_DEG, BIN_DEG)
centres = 0.5 * (edges[:-1] + edges[1:])
bin_idx = np.digitize(el, edges) - 1
ZERO = int(np.argmin(np.abs(centres)))


def to_db(x):
    with np.errstate(divide="ignore", invalid="ignore"):
        return 10.0 * np.log10(x)


def flanking(c):
    """Offsets +-3..+-6: eight non-comb channels, clear of the comb spillover.

    No FM condition, unlike the version this exploration started from. Only one
    tone in the band, 111.3 MHz, has a flanker inside the 86-110 MHz guard
    window; keeping the exception moves that one curve by 0.1 dB and costs a
    seven-channel special case in the caption.
    """
    return [c + o for o in range(-6, 7)
            if 3 <= abs(o) <= 6 and 0 <= c + o < nchan and finite[c + o]
            and (c + o) % 16 not in COMB_SPILL]


def binned(series, a, b):
    s = series[a:b]
    idx = bin_idx[a:b]
    mu = np.full(len(centres), np.nan)
    for j in range(len(centres)):
        v = s[(idx == j) & np.isfinite(s)]
        if v.size:
            mu[j] = np.median(v)
    return mu


def subtract(c):
    return d4[:, c] - np.nanmedian(d4[:, flanking(c)], axis=1)


def noise_sigma(c):
    sds = [np.nanstd(binned(subtract(cp), A, B)) for cp in flanking(c) if flanking(cp)]
    return np.nanmedian(sds) if sds else np.inf


def excess_profile(c):
    mu = binned(subtract(c), A, B)
    ok = mu > NSIG * noise_sigma(c)
    pos = mu > 0
    ref = np.nanmedian(mu[ZERO - 1:ZERO + 2])
    full = np.where(pos, to_db(np.where(pos, mu, np.nan) / ref), np.nan)
    return full, ok


def referenced(data, chans):
    """Binned power in `chans`, in dB relative to the value at 0 deg."""
    mu = binned(np.nanmedian(data[:, chans], axis=1), A, B)
    return to_db(mu / np.nanmedian(mu[ZERO - 1:ZERO + 2]))


def swing(v):
    return np.ptp(v[np.isfinite(v)])


tones = []
for c in chan[(chan % 16 == COMB_RESIDUE) & (freq > 50) & (freq < 250) & finite & ~is_fm]:
    nb = flanking(c)
    rough = np.nanmedian(np.abs(np.diff(binned(to_db(np.nanmedian(d4[:, nb], 1)), A, B), 2)))
    if rough >= MAX_ROUGHNESS:
        continue
    prof, ok = excess_profile(c)
    if not ok[ZERO] or ok.sum() < 20:
        continue
    tones.append(c)

print(f"rotation {ROT}: azimuth {rot_az[ROT]:+.0f} deg, {B - A} integrations, "
      f"{(t[B - 1] - t[A]):.0f} s")
print(f"{len(tones)} tones in panel (a), "
      f"{freq[tones[0]]:.1f}-{freq[tones[-1]]:.1f} MHz\n")

air = {c: referenced(d4, flanking(c)) for c in tones}     # suspended, rotating
gnd = {c: referenced(d0, flanking(c)) for c in tones}     # stationary control

# ------------------------------------------------- 1. where the threshold goes
print("-- ground-copy swing per tone, sorted (dB) --")
order = sorted(tones, key=lambda c: swing(gnd[c]))
vals = np.array([swing(gnd[c]) for c in order])
for c, v in zip(order, vals):
    print(f"   {freq[c]:7.1f} MHz   gnd {v:5.2f}   air {swing(air[c]):5.2f}")
jump = int(np.argmax(np.diff(vals)))
step = int(np.searchsorted(vals, 0.35))
print(f"\n   values run {vals[0]:.2f}-{vals[step - 1]:.2f} and then "
      f"{vals[step]:.2f}-{vals[-1]:.2f}: a step of "
      f"{vals[step] - vals[step - 1]:.2f} dB at the threshold.")
print(f"   Be careful how much weight that carries. It is NOT the largest break in")
print(f"   the distribution -- that is {vals[jump]:.2f} -> {vals[jump + 1]:.2f} dB, "
      f"after {jump + 1} tones -- and neighbouring")
print(f"   spacings are comparable, so 'the threshold sits in a natural gap' is more")
print(f"   than the data supports. The defence that does hold is the stability scan below.")

GND_MAX = 0.35
keep = [c for c in tones if swing(gnd[c]) < GND_MAX]
drop = [c for c in tones if swing(gnd[c]) >= GND_MAX]
kept_swing = np.array([swing(air[c]) for c in keep])
print(f"\n   GND_MAX = {GND_MAX}: {len(keep)} of {len(tones)} tones, "
      f"{freq[keep[0]]:.1f}-{freq[keep[-1]]:.1f} MHz")
print(f"   sky swing {kept_swing.min():.2f}-{kept_swing.max():.2f} dB "
      f"(median {np.median(kept_swing):.2f}, "
      f"{100 * (10 ** (np.median(kept_swing) / 10) - 1):.0f} per cent in power)")
print(f"   median ground-copy swing of the kept population is "
      f"{np.median([swing(gnd[c]) for c in keep]):.2f} dB, so the gap is only a "
      f"factor of {GND_MAX / np.median([swing(gnd[c]) for c in keep]):.1f} above it")

# ------------------------------------- 2. the quoted median against the choice
print("\n-- does the quoted median depend on the threshold? --")
print("   thr   tones   median sky swing   band")
for thr in (1.00, 0.60, 0.35, 0.25, 0.20, 0.15, 0.12):
    k = [c for c in tones if swing(gnd[c]) < thr]
    if not k:
        continue
    m = np.median([swing(air[c]) for c in k])
    print(f"   {thr:4.2f}   {len(k):5d}   {m:14.2f} dB   "
          f"{freq[k[0]]:5.1f}-{freq[k[-1]]:5.1f} MHz")
print("   The median moves by little across a factor of eight in threshold. The rise")
print("   at the tight end is frequency coverage, not cleanliness: only the low tones")
print("   survive there, and those have the deepest sky modulation anyway.")

# ------------------------------- 3. what the cut does and does not say about (a)
print("\n-- what the cut costs panel (a) --")


def margin_db(c, bins=None):
    """Tone power above its own continuum, in dB, per rotation-angle bin."""
    tone_b = binned(d4[:, c], A, B)
    cont_b = binned(np.nanmedian(d4[:, flanking(c)], axis=1), A, B)
    exc = tone_b - cont_b
    m = np.isfinite(exc) & np.isfinite(cont_b) & (exc > 0)
    if bins is not None:
        m &= bins
    return to_db(exc[m] / cont_b[m])


def worst_shift(c, margins):
    """Curve shift if the entire ground-copy swing were error on the continuum."""
    err = 10 ** (swing(gnd[c]) / 10) - 1.0
    return abs(to_db(1.0 + err / 10 ** (margins / 10))).max()


def margin_at_zero(c):
    cont_b = binned(np.nanmedian(d4[:, flanking(c)], axis=1), A, B)
    exc0 = binned(d4[:, c], A, B)[ZERO] - cont_b[ZERO]
    return to_db(exc0 / cont_b[ZERO])


peak = np.array([margin_at_zero(c) for c in drop])
print(f"   at 0 deg, the response peak, the {len(drop)} discarded tones sit "
      f"{peak.min():.1f}-{peak.max():.1f} dB above their own continuum,")
print(f"   so there the whole ground-copy swing moves those curves by at most "
      f"{max(worst_shift(c, np.array([m])) for c, m in zip(drop, peak)):.3f} dB.")

print("\n   BUT that is the most favourable point of the turn, and it is not the")
print("   number the argument needs. Over the bins panel (a) actually draws and")
print("   quotes depths from, the margin closes as the tone goes into its null:")
sig_lo, sig_shift = [], []
for c in drop:
    _, ok = excess_profile(c)
    mg = margin_db(c, bins=ok)
    if mg.size:
        sig_lo.append((freq[c], mg.min(), worst_shift(c, mg)))
sig_lo.sort(key=lambda r: r[1])
print(f"   margin over significant bins : {min(r[1] for r in sig_lo):.1f} to "
      f"{max(r[1] for r in sig_lo):.1f} dB")
print(f"   worst-case curve shift       : {max(r[2] for r in sig_lo):.2f} dB")
print("   tightest three:")
for f_, lo, sh in sig_lo[:3]:
    print(f"      {f_:7.1f} MHz   margin {lo:5.1f} dB   shift up to {sh:.2f} dB")
print("\n   So the honest claim is bounded to the top of the range: the contamination")
print("   the cut removes is negligible against panel (a)'s signal near the peak, and")
print("   is NOT negligible near the null. That is consistent with what the paper")
print("   already says -- below 158 MHz the tone reaches the noise before the null")
print("   does and those depths are reported as upper limits -- but it means the")
print("   claim must be stated at the peak, not for the curve as a whole.")

# ------------------------------------ 4. what the test cannot do: steady + fixed
print("\n-- the blind spot: a source steady in time but fixed in direction --")
print("   It is constant on a stationary receiver, so it passes, yet a rotating")
print("   antenna modulates it exactly like a point source. The aeronautical band")
print("   shows both halves, and the ground copy splits them where the allocation does:")
print("\n   band              suspended   ground copy")
for lo, hi, what in ((70.0, 86.0, "clean sky"),
                     (110.0, 118.0, "navigation beacons, continuous"),
                     (118.0, 137.0, "airband voice, intermittent"),
                     (137.0, 145.0, "back to normal")):
    sel = [c for c in tones if lo <= freq[c] < hi]
    if not sel:
        continue
    print(f"   {lo:5.0f}-{hi:5.0f} MHz   {np.median([swing(air[c]) for c in sel]):6.2f} dB"
          f"   {np.median([swing(gnd[c]) for c in sel]):6.2f} dB   {what}")
print("\n   channel by channel on the ground copy, through the transition")
print("   (the continuum channels themselves, not the tones they feed):")
band_ch = [c for c in chan if 109.0 <= freq[c] <= 122.0
           and finite[c] and c % 16 not in COMB_SPILL]
for c in band_ch:
    v = referenced(d0, [c])
    print(f"      {freq[c]:7.2f} MHz   gnd {swing(v):5.2f}")
print("   The intermittent half is dropped; the continuous half is not, which is why")
print("   the deepest curves left in the panel sit just above 110 MHz.")
low = [c for c in tones if 110.0 <= freq[c] < 118.0]
print(f"   Not one channel dragging a median: every tone from 110 to 118 MHz modulates")
print(f"   {min(swing(air[c]) for c in low):.1f}-{max(swing(air[c]) for c in low):.1f} dB, "
      f"decaying with distance from the band, so an outlier test over the eight")
print("   flankers would find nothing.")

# ----------------------------------------------------------------- the figure
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for c in keep:
    ax[0].plot(centres, air[c], color="C0", lw=0.8, alpha=0.8)
for c in drop:
    ax[0].plot(centres, air[c], color="C3", lw=0.8, alpha=0.8)
ax[0].set_xlim(-180, 180)
ax[0].set_xticks([-180, -90, 0, 90, 180])
ax[0].axhline(0.0, color="0.75", lw=0.5, ls=":")
ax[0].grid(alpha=0.25, lw=0.4)
ax[0].set_xlabel("Platform rotation angle [deg]")
ax[0].set_ylabel("Continuum estimate rel. $0^\\circ$ [dB]")
ax[0].set_title(f"blue: kept ({len(keep)})   red: ground copy moved too ({len(drop)})",
                fontsize=9)

ax[1].semilogy(freq[tones], [swing(gnd[c]) for c in tones], "o", ms=4, color="0.3")
ax[1].axhline(GND_MAX, color="C3", lw=1.0, ls="--")
ax[1].text(60, GND_MAX * 1.1, f"GND_MAX = {GND_MAX} dB", fontsize=8, color="C3")
ax[1].axvspan(110, 137, color="C1", alpha=0.15)
ax[1].text(112, 0.045, "aeronautical", fontsize=8, color="C1")
ax[1].grid(alpha=0.25, lw=0.4, which="both")
ax[1].set_xlabel("Frequency [MHz]")
ax[1].set_ylabel("Ground-copy swing over the turn [dB]")

fig.tight_layout()
fig.savefig(OUT / "61_ground_copy_cut.png", dpi=140)
print(f"\nsaved {OUT / '61_ground_copy_cut.png'}")
